# ASI03 Identity & Privilege Abuse — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI03 — Identity & Privilege Abuse | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01 (Prompt Injection), LLM02 (Sensitive Info Disclosure), LLM06 (Excessive Agency)

**ASI03 tests for**:
- Privilege escalation via impersonation (claiming admin/exec authority without credentials)
- Lateral scope expansion (accessing other users' data via social engineering)
- Agent identity spoofing (sub-agent messages containing trust-escalation payloads)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
from pathlib import Path

_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}
for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    name = f"ASI03-{jsonl_path.stem}"
    sc = okareo.upload_scenario_set(scenario_name=name, file_path=str(jsonl_path))
    registered_scenarios[name] = sc
    print(f"  ✓ {name} (ID: {sc.scenario_id})")

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}
for md_path in sorted(checks_dir.glob("*.md")):
    data = parse_artifact(md_path)
    result = okareo.create_or_update_check(
        name=data["name"], description=data["description"],
        check=ModelBasedCheck(prompt_template=data["prompt_template"], check_type=CheckOutputType.PASS_FAIL),
    )
    registered_checks[data["name"]] = result.id
    print(f"  ✓ {data['name']} (ID: {result.id})")

In [ ]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}
for md_path in sorted(drivers_dir.glob("*.md")):
    data = parse_artifact(md_path, default_temperature=0.7)
    result = okareo.create_or_update_driver(driver=Driver(
        name=data["name"], prompt_template=data["prompt_template"], temperature=data["temperature"]
    ))
    registered_drivers[data["name"]] = result
    print(f"  ✓ {data['name']} (ID: {result.id})")

---
## Part 2 — Run Evaluation

In [ ]:
target = build_target(CATEGORY_DIR)
PRIV_CHECK = "ASI03-privilege-abuse-detector"
MAX_TURNS = 10
SINGLE_TURN_SCENARIOS = [
    "ASI03-privilege-escalation-by-impersonation",
    "ASI03-lateral-scope-expansion",
    "ASI03-agent-identity-spoofing",
]
MULTI_TURN_MAP = {
    "ASI03-privilege-escalation-by-impersonation": "ASI03-impersonation-escalator",
}
all_results = {}

In [ ]:
st_driver = Driver(name="pass-through", prompt_template=SINGLE_TURN_DRIVER_TEMPLATE, temperature=0)
for sname in SINGLE_TURN_SCENARIOS:
    if sname not in registered_scenarios: continue
    try:
        run = okareo.run_simulation(
            target=target, driver=st_driver,
            name=f"ASI03 Single-Turn — {sname}",
            api_key=OKAREO_API_KEY, first_turn="target",
            scenario=registered_scenarios[sname], max_turns=1, checks=[PRIV_CHECK],
        )
        all_results[sname] = run
        print(f"  ✓ {sname}: {getattr(run, 'app_link', run.id)}")
    except Exception as e:
        all_results[sname] = None; print(f"  ✗ {sname}: {e}")

In [ ]:
for sname, dname in MULTI_TURN_MAP.items():
    if sname not in registered_scenarios or dname not in registered_drivers: continue
    drv = registered_drivers[dname]
    try:
        run = okareo.run_simulation(
            target=target,
            driver=Driver(temperature=getattr(drv,"temperature",0.7), name=dname, prompt_template=drv.prompt_template),
            name=f"ASI03 Simulation — {sname}",
            api_key=OKAREO_API_KEY, first_turn="target",
            scenario=registered_scenarios[sname], max_turns=MAX_TURNS, checks=[PRIV_CHECK],
        )
        key = f"{sname} (multi-turn)"
        all_results[key] = run
        print(f"  ✓ {key}: {getattr(run, 'app_link', run.id)}")
    except Exception as e:
        all_results[f"{sname} (multi-turn)"] = None; print(f"  ✗ {e}")

In [ ]:
print("\n" + "=" * 60)
print("ASI03 IDENTITY & PRIVILEGE ABUSE — EVALUATION RESULTS")
print("=" * 60)
for name, result in all_results.items():
    status = getattr(result, "app_link", None) or (result.id if result else "ERROR")
    icon = "✓" if result else "✗"
    print(f"  {icon} {name}: {status}")
errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal: {len(all_results)} | Errors: {errors}")